# Transfer learning with torchvision.models

_PyTorch (a complete course)_

**Start from a model pretrained on millions of images, then teach it your task with very little data.**

A deep image network learns its features in layers: early layers detect edges and colors, middle layers detect textures and parts, late layers detect whole objects. The early and middle features are generic — useful for almost any image task. Transfer learning keeps those learned layers (the backbone) and only replaces the last layer (the head) that maps features to your specific classes.

---

This notebook is a practice scaffold for the **Transfer learning with torchvision.models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 5   # <- your number of classes

# ---- 1. LOAD A PRETRAINED MODEL ----
weights = ResNet18_Weights.DEFAULT          # best available ImageNet weights
model = resnet18(weights=weights)           # downloads architecture + trained weights

# The weights object carries the EXACT preprocessing the model expects.
# Apply this to every input image (resize, center-crop, ImageNet normalization).
preprocess = weights.transforms()
print("expected transforms:\n", preprocess)

# ---- 2. FREEZE THE BACKBONE (feature-extraction mode) ----
for param in model.parameters():
    param.requires_grad = False             # autograd will not update these

# ---- 3. SWAP THE FINAL LAYER FOR A NEW HEAD ----
in_features = model.fc.in_features          # 512 for resnet18
model.fc = nn.Linear(in_features, NUM_CLASSES)   # fresh layer -> requires_grad=True
model = model.to(device)

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable parameters:", trainable)   # only 'fc.weight', 'fc.bias'

# ---- 4. OPTIMIZER SEES ONLY THE HEAD ----
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)   # NOT model.parameters()
criterion = nn.CrossEntropyLoss()           # expects raw logits + class indices

# ---- 5. A SHORT TRAINING LOOP ----
# Tiny synthetic 'dataset' so this runs anywhere. resnet18 wants 3x224x224 inputs.
# In a real project: datasets.ImageFolder(root, transform=preprocess) + DataLoader.
x = torch.randn(16, 3, 224, 224, device=device)         # 16 fake images
y = torch.randint(0, NUM_CLASSES, (16,), device=device) # 16 fake labels

model.eval()        # backbone batchnorm uses stored running stats (frozen)
model.fc.train()    # but the head is training
for epoch in range(5):
    optimizer.zero_grad()           # clear last step's grads (they accumulate!)
    logits = model(x)               # forward pass
    loss = criterion(logits, y)     # cross-entropy on logits (no softmax needed)
    loss.backward()                 # grads flow only into the head
    optimizer.step()                # update the head
    print(f"epoch {epoch}: loss = {loss.item():.4f}")

# ---- 6. SWITCH TO FULL FINE-TUNING (optional phase two) ----
# Unfreeze the backbone and train everything with a SMALL learning rate so the
# good pretrained weights only adjust gently.
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)   # small lr for fine-tuning
model.train()
optimizer.zero_grad()
loss = criterion(model(x), y)
loss.backward()
optimizer.step()
print("after one fine-tune step:", loss.item())


## Visualize the data & results

_Transfer learning should win most when labels are scarce. Using frozen learned features (a stand-in for a pretrained backbone) vs raw pixels, how does test accuracy on a real face dataset grow as we give the classifier more labeled images?_

In [ ]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Real images: 400 face photos of 40 people, 64x64 = 4096 raw pixels each.
faces = fetch_olivetti_faces()
X, y = faces.data, faces.target

# Hold out a fixed test set; sample tiny label budgets from the rest.
X_pool, X_te, y_pool, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)

# TRANSFER (feature-extraction mode): a FROZEN backbone. An eigenface PCA basis
# learned once on the face manifold stands in for a pretrained backbone's frozen
# features; we train only a linear head on the few labels.
center = StandardScaler(with_std=False).fit(X_pool)
backbone = PCA(n_components=100, whiten=True, random_state=0).fit(center.transform(X_pool))
def features(Z):
    return backbone.transform(center.transform(Z))

classes = np.unique(y_pool)
def sample(per_class, seed):
    rng = np.random.RandomState(seed); idx = []
    for c in classes:
        ci = np.where(y_pool == c)[0]
        idx.extend(rng.choice(ci, size=min(per_class, len(ci)), replace=False))
    return np.array(idx)

def accuracy(per_class, mode):
    accs = []
    for seed in range(20):                       # average over 20 random samples
        idx = sample(per_class, seed)
        Xs, ys = X_pool[idx], y_pool[idx]
        if mode == "transfer":                   # frozen features + linear head
            Ftr, Fte = features(Xs), features(X_te)
        else:                                    # FROM SCRATCH: raw pixels, no learned features
            Ftr, Fte = Xs, X_te
        clf = LogisticRegression(max_iter=3000, C=1e4).fit(Ftr, ys)
        accs.append(clf.score(Fte, y_te))
    return round(float(np.mean(accs)), 3)

budgets = [1, 2, 3, 4, 5, 7]                      # labeled images PER PERSON
total = [b * 40 for b in budgets]                # 40 people
print("labeled images total:", total)
print("transfer    :", [accuracy(b, "transfer") for b in budgets])
print("from scratch :", [accuracy(b, "scratch") for b in budgets])
# labeled images total: [40, 80, 120, 160, 200, 280]
# transfer    : [0.548, 0.707, 0.815, 0.858, 0.893, 0.925]
# from scratch : [0.506, 0.655, 0.727, 0.761, 0.778, 0.85]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You load a pretrained resnet18, replace model.fc with nn.Linear(512, 3), and create optim.Adam(model.parameters(), lr=1e-3). Training is slow and the model overfits your 200 images. You intended feature extraction. What two things are wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether the backbone was frozen. — _Nothing set requires_grad = False, so every layer is still trainable._
- Check what you handed the optimizer. — _model.parameters() is the whole network, not just the head._

**Answer:** Two fixes. First, freeze the backbone: for p in model.parameters(): p.requires_grad = False before swapping the head (the new head is created trainable). Second, give the optimizer only the head: optim.Adam(model.fc.parameters(), lr=1e-3). Now you train a small classifier on fixed features — fast and far less prone to overfitting on 200 images.

</details>

**Problem 2.** Your transfer model trains fine but test accuracy is far worse than expected. You loaded the weights correctly and the head is the right size. You preprocessed images by simply dividing pixel values by 255. What is the likely cause?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how the pretrained model was trained. — _It expects a specific resize, crop, and per-channel normalization._
- Compare your preprocessing to the model's expected transforms. — _Dividing by 255 is not the ImageNet normalization the weights were trained with._

**Answer:** You skipped the model's expected input transforms. Use tf = weights.transforms() and apply it to every image. It does the right resize/center-crop and subtracts the ImageNet per-channel mean and divides by its standard deviation — the exact distribution the weights expect. Mismatched normalization shifts the inputs off the manifold the network learned, so accuracy drops.

</details>

**Problem 3.** After training the head, you unfreeze the whole backbone and continue with learning rate 1e-2. Accuracy suddenly gets much worse. Why, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Think about how big each gradient step is at lr = 1e-2. — _That is a large step for weights that are already near-optimal._
- Consider what large updates do to pretrained weights. — _They overwrite the useful features before the model can gently adapt._

**Answer:** The learning rate is too high for fine-tuning. The pretrained weights are already good, so big steps wreck them. Drop to a small rate like 1e-4 or 1e-5 (often smaller than the head's rate) so the backbone adapts gently. A common pattern is two phases: train the head at a moderate rate, then unfreeze and fine-tune everything at a much lower rate.

</details>